# 대조학습 문장임베딩 앙상블 — 원문제 모범답안2 코드 그대로 (Colab)

**연습 기법** (IOAI 2025 *at-home* **Chameleon** *모범답안2*(최고점 0.649→**0.724**) 와 동일): 사전학습 문장임베딩을
**MNR 대조학습 + 하드 네거티브**로 미세조정하고, 서로 다른 모델을 **z-norm 앙상블**한다. **이 노트북 코드는
Chameleon 모범답안2 골격을 그대로** 따른다:

| Chameleon 모범답안2 | 이 노트북 |
|---|---|
| `MEMBERS` (bge-large, e5-instruct) | `MEMBERS` (bge-small, e5-small) |
| `make_examples` → `InputExample([anchor, pos] + hard_negs)` | 동일 (anchor, pos, hard_neg) |
| `train_member` → `losses.MultipleNegativesRankingLoss` + `m.fit` | 동일 (그대로) |
| `_znorm` + 앙상블 유사도 → 옵션 선택 | `_znorm` + 앙상블 → 클래스(2지선다) 선택 |

**데이터**: [NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started) (재난 트윗 이진분류). Chameleon 은
*힌트→정답단어(100지선다 검색)* 라 옵션이 100개지만, 여기선 *트윗→재난/일반(2지선다)* 이라 **클래스 중심(centroid)
2개를 옵션**처럼 쓴다(과제 형태 차이라 추론만 다름, 미세조정 코드는 동일).

> ⚠️ z-norm 적용 축 차이: Chameleon 은 *쿼리당 100옵션 사이* z-norm, 여기선 *모델당 샘플분포* z-norm(이진은 옵션이 2개라
> 옵션 축 z-norm 이 무의미 → 샘플축 보정이 올바른 적응). 미세조정(MNR+하드네거티브)은 모범답안과 동일.
> ⚙️ GPU 권장(작은 임베딩 모델, T4 수 분).  ⚠️ API 토큰 평문 — 외부 공유 금지. 12번(BERT 분류헤드)과 접근이 다름.

## 0. 설치 · 자격증명 · 시드

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "sentence-transformers", "scikit-learn", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
import random, numpy as np, torch
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
print("준비 완료")

## 1. Kaggle 에서 Disaster Tweets 다운로드 & train/val 분할

In [ ]:
import glob, zipfile, pandas as pd
from sklearn.model_selection import train_test_split
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("nlp-getting-started", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
train = pd.read_csv(os.path.join(WORK_DIR, "train.csv")); test = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
train["text"] = train["text"].fillna(""); test["text"] = test["text"].fillna("")
tr_idx, va_idx = train_test_split(np.arange(len(train)), test_size=0.15, random_state=42, stratify=train["target"])
tr_text = train["text"].values[tr_idx]; tr_y = train["target"].values[tr_idx]
val_text = train["text"].values[va_idx]; val_y = train["target"].values[va_idx]
device = "cuda" if torch.cuda.is_available() else "cpu"
print("train:", len(tr_text), "| val:", len(val_text), "| test:", len(test))

## 2. 미세조정 — make_examples(하드네거티브) + train_member(MNR) (모범답안2 그대로)
`make_examples` 는 `InputExample([anchor, positive, hard_neg])` 삼중쌍을 만든다(하드 네거티브 = 반대 클래스 중
베이스 임베딩이 가장 비슷한 샘플). `train_member` 는 `MultipleNegativesRankingLoss` 로 `m.fit`.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import DataLoader

# 원문제의 bge-large + e5-instruct 를 T4 친화적 small 변형으로 1:1 대응
MEMBERS = [{"model": "BAAI/bge-small-en-v1.5", "qp": ""},
           {"model": "intfloat/e5-small-v2",   "qp": "query: "}]
rng = np.random.RandomState(0)

def make_examples(model, qp):
    texts = [qp + t for t in tr_text]
    base = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    examples = []
    for cls in (0, 1):
        same = np.where(tr_y == cls)[0]; opp = np.where(tr_y != cls)[0]
        nn = NearestNeighbors(n_neighbors=1, metric="cosine").fit(base[opp])
        _, nbr = nn.kneighbors(base[same])
        for k, i in enumerate(same):
            j = i
            while j == i and len(same) > 1: j = same[rng.randint(len(same))]   # 같은 클래스 positive
            hard = opp[nbr[k, 0]]                                              # 반대 클래스 최근접 = 하드 네거티브
            examples.append(InputExample(texts=[texts[i], texts[j], texts[hard]]))
    return examples

def train_member(mb, epochs=1, batch_size=32):
    m = SentenceTransformer(mb["model"], device=device)
    dl = DataLoader(make_examples(m, mb["qp"]), shuffle=True, batch_size=batch_size)
    loss = losses.MultipleNegativesRankingLoss(m)
    m.fit(train_objectives=[(dl, loss)], epochs=epochs, warmup_steps=int(0.1 * len(dl)), show_progress_bar=True)
    print(f"  \u2713 {mb['model']}: {len(dl.dataset)}\uac1c \uc0bc\uc911\uc30d {epochs}ep \ubbf8\uc138\uc870\uc815 \uc644\ub8cc")
    return m

tuned = [(train_member(mb), mb) for mb in MEMBERS]

## 3. z-norm 앙상블 추론 (모범답안2 `_znorm` / 앙상블 유사도 대응)
각 멤버: train 임베딩으로 **클래스 중심(centroid) 2개**(=옵션)를 만들고 `재난성 점수 = cos(x,중심1)−cos(x,중심0)`.
모델별로 **z-norm**(샘플축, train 통계) 후 평균 = 앙상블. 점수>0 이면 재난(2지선다 argmax).

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def _znorm(a, mu, sd): return (a - mu) / (sd + 1e-8)

def member_score(m, qp):
    enc = lambda txts: m.encode([qp + t for t in txts], batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    e_tr = enc(tr_text)
    c1 = e_tr[tr_y == 1].mean(0); c1 /= np.linalg.norm(c1)     # 재난 중심 (옵션 1)
    c0 = e_tr[tr_y == 0].mean(0); c0 /= np.linalg.norm(c0)     # 일반 중심 (옵션 0)
    f = lambda E: E @ c1 - E @ c0
    return f(e_tr), f(enc(val_text)), f(enc(test["text"].values))

scores = [member_score(m, mb["qp"]) for m, mb in tuned]

def ensemble(which):                                          # which: 0=train,1=val,2=test
    return np.mean([_znorm(s[which], s[0].mean(), s[0].std()) for s in scores], axis=0)

z_val, z_test = ensemble(1), ensemble(2)
val_pred = (z_val > 0).astype(int)                            # 점수>0 → 재난 (argmax)
print(f"앙상블  val accuracy = {accuracy_score(val_y, val_pred):.4f} | val F1 = {f1_score(val_y, val_pred):.4f}")
for (m, mb), s in zip(tuned, scores):
    z = _znorm(s[1], s[0].mean(), s[0].std())
    print(f"  \ub2e8\uc77c {mb['model']:24s} val F1 = {f1_score(val_y, (z > 0).astype(int)):.4f}")
print("\u2192 MNR+\ud558\ub4dc\ub124\uac70\ud2f0\ube0c \ubbf8\uc138\uc870\uc815 + z-norm \uc559\uc0c1\ube14 \u2014 Chameleon \ubaa8\ubc94\ub2f5\uc5482 \uc758 \ud575\uc2ec \uae30\ubc95 \uadf8\ub300\ub85c.")

## 4. 캐글 제출 파일 생성 (`id,target`)

In [ ]:
test_pred = (z_test > 0).astype(int)
submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"id": test["id"], "target": test_pred}).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| 1\uc758 \ube44\uc728:", round(test_pred.mean(), 3))
pd.read_csv(submission_path).head()

In [ ]:
try:
    from google.colab import files; files.download(submission_path)
except Exception:
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started/submit) 에 제출.

Chameleon **모범답안2**(최고점) 골격(`MEMBERS`→`make_examples`(하드네거티브)→`train_member`(**MNR**+`m.fit`)→`_znorm`
앙상블→옵션 선택)을 그대로 옮겼다. 핵심은 *MNR 대조학습+하드네거티브 미세조정 후 z-norm 앙상블*. 과제가 이진분류라
추론만 2지선다(centroid)로 적응. 더: 에폭↑, 하드네거티브 개수↑, 더 큰 모델(bge-base/large·e5-base).